In [6]:
import sys
import numpy as np
import pandas as pd

import torch
import torch.backends.cudnn as cudnn
cudnn.enabled = False  # avoid LSTM CuDNN weirdness

import lightning.pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping

from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting.models import TemporalFusionTransformer as TFTModel
from pytorch_forecasting.metrics import MAE

pl.seed_everything(42, workers=True)

H4_PATH = r"C:\Users\admin\Desktop\Forex_algo\code\Data\EUR_USD_H4.parquet"
print("H4 path:", H4_PATH)


c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\pytorch_forecasting\models\base\_base_model.py:28: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm
Seed set to 42


H4 path: C:\Users\admin\Desktop\Forex_algo\code\Data\EUR_USD_H4.parquet


In [7]:
df_h4 = pd.read_parquet(H4_PATH)
print("Raw H4 shape:", df_h4.shape)
print("Columns:", df_h4.columns.tolist())

df_h4["time"] = pd.to_datetime(df_h4["time"])
df_h4 = df_h4.sort_values("time").reset_index(drop=True)

df_h4.head()


Raw H4 shape: (15375, 14)
Columns: ['time', 'volume', 'mid_o', 'mid_h', 'mid_l', 'mid_c', 'bid_o', 'bid_h', 'bid_l', 'bid_c', 'ask_o', 'ask_h', 'ask_l', 'ask_c']


,time,volume,mid_o,mid_h,mid_l,mid_c,bid_o,bid_h,bid_l,bid_c,ask_o,ask_h,ask_l,ask_c
0,2016-01-06 22:00:00+00:00,5015,1.07807,1.08100,1.07737,1.08029,1.07778,1.08092,1.07724,1.08020,1.07836,1.08109,1.07747,1.08038
1,2016-01-07 02:00:00+00:00,3448,1.08026,1.08297,1.07996,1.08272,1.08018,1.08288,1.07987,1.08265,1.08035,1.08306,1.08005,1.08279
2,2016-01-07 06:00:00+00:00,11556,1.08274,1.08698,1.07713,1.08460,1.08267,1.08688,1.07703,1.08447,1.08281,1.08707,1.07723,1.08473
3,2016-01-07 10:00:00+00:00,8239,1.08458,1.08748,1.08350,1.08588,1.08447,1.08739,1.08342,1.08581,1.08469,1.08757,1.08358,1.08595
4,2016-01-07 14:00:00+00:00,10399,1.08584,1.08751,1.08263,1.08626,1.08577,1.08744,1.08255,1.08618,1.08591,1.08758,1.08271,1.08633


In [9]:
# Cell 3: Add indicators to H4 candles

project_root = r"C:\Users\admin\Desktop\Forex_algo\code"
if project_root not in sys.path:
    sys.path.append(project_root)

from technicals.indicators import BollingerBands, ATR, KeltnerChannels, RSI, MACD

print("Using indicators from:", project_root)

df = df_h4.copy()

# 1) Your indicators
df = BollingerBands(df)       # -> BB_MA, BB_UP, BB_LW
df = ATR(df, n=14)            # -> ATR_14
df = KeltnerChannels(df)      # -> EMA, KeUp, KeLo
df = RSI(df, n=14)            # -> RSI_14
df = MACD(df)                 # -> MACD, SIGNAL, HIST

# 2) Aliases with clean names
df["rsi"]          = df["RSI_14"]
df["macd"]         = df["MACD"]
df["macd_signal"]  = df["SIGNAL"]
df["macd_hist"]    = df["HIST"]
df["bb_middle"]    = df["BB_MA"]
df["bb_upper"]     = df["BB_UP"]
df["bb_lower"]     = df["BB_LW"]
df["atr14"]        = df["ATR_14"]

# 3) EMAs on H4 mid_c
df["ema_5"]   = df["mid_c"].ewm(span=5,   min_periods=5).mean()
df["ema_20"]  = df["mid_c"].ewm(span=20,  min_periods=20).mean()
df["ema_50"]  = df["mid_c"].ewm(span=50,  min_periods=50).mean()
df["ema_200"] = df["mid_c"].ewm(span=200, min_periods=200).mean()

# 4) Engineered features
df["momentum_oc"]        = df["mid_o"] - df["mid_c"]
df["avg_price_hl"]       = (df["mid_h"] + df["mid_l"]) / 2.0
df["range_hl"]           = df["mid_h"] - df["mid_l"]
df["typical_price_ohlc"] = (df["mid_o"] + df["mid_h"] + df["mid_l"] + df["mid_c"]) / 4.0

# 5) Log volume
df["log_volume"] = np.log1p(df["volume"])

print("After indicators:", df.shape)
df.head()


Using indicators from: C:\Users\admin\Desktop\Forex_algo\code
After indicators: (15375, 42)


,time,volume,mid_o,mid_h,mid_l,mid_c,bid_o,bid_h,bid_l,bid_c,...,atr14,ema_5,ema_20,ema_50,ema_200,momentum_oc,avg_price_hl,range_hl,typical_price_ohlc,log_volume
0,2016-01-06 22:00:00+00:00,5015,1.07807,1.08100,1.07737,1.08029,1.07778,1.08092,1.07724,1.08020,...,NaN,NaN,NaN,NaN,NaN,-0.00222,1.079185,0.00363,1.079182,8.520388
1,2016-01-07 02:00:00+00:00,3448,1.08026,1.08297,1.07996,1.08272,1.08018,1.08288,1.07987,1.08265,...,NaN,NaN,NaN,NaN,NaN,-0.00246,1.081465,0.00301,1.081478,8.145840
2,2016-01-07 06:00:00+00:00,11556,1.08274,1.08698,1.07713,1.08460,1.08267,1.08688,1.07703,1.08447,...,NaN,NaN,NaN,NaN,NaN,-0.00186,1.082055,0.00985,1.082863,9.355047
3,2016-01-07 10:00:00+00:00,8239,1.08458,1.08748,1.08350,1.08588,1.08447,1.08739,1.08342,1.08581,...,NaN,NaN,NaN,NaN,NaN,-0.00130,1.085490,0.00398,1.085360,9.016756
4,2016-01-07 14:00:00+00:00,10399,1.08584,1.08751,1.08263,1.08626,1.08577,1.08744,1.08255,1.08618,...,NaN,1.085024,NaN,NaN,NaN,-0.00042,1.085070,0.00488,1.085560,9.249561


In [10]:
# Cell 4: Target = next H4 log-return

df["target_return_4h"] = np.log(df["mid_c"].shift(-1)) - np.log(df["mid_c"])

# Drop rows with NaN (indicators warmup + last bar)
df = df.dropna().reset_index(drop=True)

print("Shape after target & dropna:", df.shape)
df[["time", "mid_c", "target_return_4h"]].head(10)


Shape after target & dropna: (15175, 43)


,time,mid_c,target_return_4h
0,2016-02-23 02:00:00+00:00,1.10370,-0.003085
1,2016-02-23 06:00:00+00:00,1.10030,0.000709
2,2016-02-23 10:00:00+00:00,1.10108,0.000944
3,2016-02-23 14:00:00+00:00,1.10212,-0.000181
4,2016-02-23 18:00:00+00:00,1.10192,-0.001580
5,2016-02-23 22:00:00+00:00,1.10018,0.001181
6,2016-02-24 02:00:00+00:00,1.10148,-0.002919
7,2016-02-24 06:00:00+00:00,1.09827,0.001429
8,2016-02-24 10:00:00+00:00,1.09984,0.002343
9,2016-02-24 14:00:00+00:00,1.10242,-0.000971


In [11]:
# Cell 5: time_idx, series_id, hour, day_of_week

df["time_idx"] = np.arange(len(df), dtype=int)
df["series_id"] = "eurusd_h4"

df["hour"] = df["time"].dt.hour.astype(str)
df["day_of_week"] = df["time"].dt.dayofweek.astype(str)

df[["time", "time_idx", "series_id", "hour", "day_of_week"]].head()


,time,time_idx,series_id,hour,day_of_week
0,2016-02-23 02:00:00+00:00,0,eurusd_h4,2,1
1,2016-02-23 06:00:00+00:00,1,eurusd_h4,6,1
2,2016-02-23 10:00:00+00:00,2,eurusd_h4,10,1
3,2016-02-23 14:00:00+00:00,3,eurusd_h4,14,1
4,2016-02-23 18:00:00+00:00,4,eurusd_h4,18,1


In [12]:
# Cell 6: feature columns

FEATURE_COLS_EXT = [
    # Price & volume
    "mid_o", "mid_h", "mid_l", "mid_c",
    "volume", "log_volume",

    # Indicators
    "rsi",
    "macd", "macd_signal", "macd_hist",
    "atr14",
    "bb_lower", "bb_middle", "bb_upper",
    "ema_5", "ema_20", "ema_50", "ema_200",

    # Engineered
    "momentum_oc", "avg_price_hl", "range_hl", "typical_price_ohlc",
]

print("FEATURE_COLS_EXT:", len(FEATURE_COLS_EXT))
FEATURE_COLS_EXT


FEATURE_COLS_EXT: 22


['mid_o',
 'mid_h',
 'mid_l',
 'mid_c',
 'volume',
 'log_volume',
 'rsi',
 'macd',
 'macd_signal',
 'macd_hist',
 'atr14',
 'bb_lower',
 'bb_middle',
 'bb_upper',
 'ema_5',
 'ema_20',
 'ema_50',
 'ema_200',
 'momentum_oc',
 'avg_price_hl',
 'range_hl',
 'typical_price_ohlc']

In [13]:
# Cell 7: train / val / test split by time

df = df.sort_values("time_idx").reset_index(drop=True)

N = len(df)
train_end = int(N * 0.7)
val_end   = int(N * 0.85)

train_df = df.iloc[:train_end].copy()
val_df   = df.iloc[train_end:val_end].copy()
test_df  = df.iloc[val_end:].copy()

print("Total:", N)
print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))
print("Train time_idx:", train_df.time_idx.min(), "->", train_df.time_idx.max())
print("Val   time_idx:", val_df.time_idx.min(),   "->", val_df.time_idx.max())
print("Test  time_idx:", test_df.time_idx.min(),  "->", test_df.time_idx.max())


Total: 15175
Train: 10622 Val: 2276 Test: 2277
Train time_idx: 0 -> 10621
Val   time_idx: 10622 -> 12897
Test  time_idx: 12898 -> 15174


In [14]:
# Cell 8: TimeSeriesDataSet (H4 horizon = 1 bar, but each bar = 4h)

MAX_ENCODER_LENGTH = 96   # 96 * 4h = 384h of context (~16 days)
MAX_PREDICTION_LENGTH = 1
BATCH_SIZE = 128

training = TimeSeriesDataSet(
    train_df,
    time_idx="time_idx",
    target="target_return_4h",
    group_ids=["series_id"],

    max_encoder_length=MAX_ENCODER_LENGTH,
    max_prediction_length=MAX_PREDICTION_LENGTH,

    time_varying_unknown_reals=FEATURE_COLS_EXT,
    time_varying_known_categoricals=["hour", "day_of_week"],
    static_categoricals=["series_id"],

    target_normalizer=None,
    allow_missing_timesteps=True,
    add_relative_time_idx=True,
    add_encoder_length=True,
    add_target_scales=False,
)

validation = TimeSeriesDataSet.from_dataset(
    training,
    data=val_df,
    stop_randomization=True,
)

test = TimeSeriesDataSet.from_dataset(
    training,
    data=test_df,
    stop_randomization=True,
)

print("Samples:")
print(" training   :", len(training))
print(" validation :", len(validation))
print(" test       :", len(test))


Samples:
 training   : 10526
 validation : 2180
 test       : 2181


In [15]:
# Cell 9: dataloaders

train_loader = training.to_dataloader(
    train=True,
    batch_size=BATCH_SIZE,
    num_workers=0,
)

val_loader = validation.to_dataloader(
    train=False,
    batch_size=BATCH_SIZE,
    num_workers=0,
)

test_loader = test.to_dataloader(
    train=False,
    batch_size=BATCH_SIZE,
    num_workers=0,
)

print("Batches:")
print(" train:", len(train_loader))
print(" val  :", len(val_loader))
print(" test :", len(test_loader))


Batches:
 train: 82
 val  : 18
 test : 18


In [16]:
# Cell 10: TFT model

hidden_size = 32
attention_heads = 4
dropout = 0.10
hidden_continuous_size = 16
lstm_layers = 2
LEARNING_RATE = 1e-3

tft_h4 = TFTModel.from_dataset(
    training,
    learning_rate=LEARNING_RATE,
    hidden_size=hidden_size,
    lstm_layers=lstm_layers,
    attention_head_size=attention_heads,
    dropout=dropout,
    hidden_continuous_size=hidden_continuous_size,
    loss=MAE(),          # MAE on H4 log-returns
    output_size=1,       # one H4 step ahead (next 4h)
    log_interval=50,
    log_val_interval=1,
    reduce_on_plateau_patience=4,
)

print("Parameters:", sum(p.numel() for p in tft_h4.parameters()))


Parameters: 128524


c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


In [17]:
# Cell 11: Trainer

checkpoint_cb = ModelCheckpoint(
    monitor="val_loss",
    mode="min",
    save_top_k=1,
    filename="eurusd_h4_tft-{epoch:02d}-{val_loss:.6f}",
)

earlystop_cb = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=8,
    min_delta=1e-5,
)

accelerator = "gpu" if torch.cuda.is_available() else "cpu"

trainer = pl.Trainer(
    max_epochs=80,
    accelerator=accelerator,
    devices=1,
    gradient_clip_val=0.1,
    precision=32,
    callbacks=[checkpoint_cb, earlystop_cb],
    log_every_n_steps=20,
)

trainer


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores


In [18]:
# Cell 12: Train H4 model

trainer.fit(
    model=tft_h4,
    train_dataloaders=train_loader,
    val_dataloaders=val_loader,
)

print("Best checkpoint path:", checkpoint_cb.best_model_path)


You are using a CUDA device ('NVIDIA GeForce RTX 4070 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name                               | Type                            | Params | Mode 
------------------------------------------------------------------------------------------------
0  | loss                               | MAE                             | 0      | train
1  | logging_metrics                    | ModuleList                      | 0      | train
2  | input_embeddings                   | MultiEmbedding                  | 97     | train
3  | prescalers                         | ModuleDict                      | 768    | train
4  | static_variable_selection

Sanity Checking DataLoader 0:   0%|          | 0/2 [00:00<?, ?it/s]

c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=21` in the `DataLoader` to improve performance.


c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=21` in the `DataLoader` to improve performance.


Epoch 28: 100%|██████████| 82/82 [00:28<00:00,  2.90it/s, v_num=57, train_loss_step=0.00163, val_loss=0.00181, train_loss_epoch=0.00205]
Best checkpoint path: c:\Users\admin\Desktop\Forex_algo\code\lightning_logs\version_57\checkpoints\eurusd_h4_tft-epoch=20-val_loss=0.001522.ckpt


In [19]:
# Cell 13: Evaluate on test set

best_model_h4 = TFTModel.load_from_checkpoint(checkpoint_cb.best_model_path)

preds_raw = best_model_h4.predict(test_loader)  # [N, 1]
preds = preds_raw.detach().cpu().reshape(-1)

all_targets = []
for batch in test_loader:
    x, y_tuple = batch  # (x, (y, weight))
    y = y_tuple[0]
    all_targets.append(y.detach().cpu().reshape(-1))

targets = torch.cat(all_targets)

preds_np = preds.numpy()
targets_np = targets.numpy()

mae_h4 = np.mean(np.abs(preds_np - targets_np))
print(f"Test MAE (H4 log-return units): {mae_h4:.8f}")

pred_dir = np.sign(preds_np)
true_dir = np.sign(targets_np)
dir_acc_h4 = (pred_dir == true_dir).mean()
print(f"Direction accuracy (H4): {dir_acc_h4:.4f} ({dir_acc_h4*100:.2f}%)")

print("\nSample comparison (first 5):")
for i in range(5):
    print(f"{i}: pred={preds_np[i]:+0.6f}, true={targets_np[i]:+0.6f}")


c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\trainer\c

Test MAE (H4 log-return units): 0.00164022
Direction accuracy (H4): 0.5044 (50.44%)

Sample comparison (first 5):
0: pred=+0.000247, true=+0.000645
1: pred=+0.000260, true=+0.001055
2: pred=+0.000728, true=-0.000271
3: pred=+0.000560, true=+0.003048
4: pred=+0.003055, true=+0.000846


In [23]:
best_ckpt = checkpoint_cb.best_model_path
best_ckpt


'c:\\Users\\admin\\Desktop\\Forex_algo\\code\\lightning_logs\\version_57\\checkpoints\\eurusd_h4_tft-epoch=20-val_loss=0.001522.ckpt'

In [24]:
from pytorch_forecasting.models import TemporalFusionTransformer

CKPT_PATH = r"c:\Users\admin\Desktop\Forex_algo\code\lightning_logs\version_57\checkpoints\eurusd_h4_tft-epoch=20-val_loss=0.001522.ckpt"

tft_h4 = TemporalFusionTransformer.load_from_checkpoint(CKPT_PATH)
print("Loaded H4 model from checkpoint:", CKPT_PATH)


Loaded H4 model from checkpoint: c:\Users\admin\Desktop\Forex_algo\code\lightning_logs\version_57\checkpoints\eurusd_h4_tft-epoch=20-val_loss=0.001522.ckpt


In [25]:
import torch
from pathlib import Path

SAVE_PATH = Path(r"C:\Users\admin\Desktop\Forex_algo\code\tft_h4-50.44.ckpt")

torch.save(
    {
        "state_dict": tft_h4.state_dict(),
        "hyper_parameters": tft_h4.hparams
    },
    SAVE_PATH
)

print("Saved compact H4 model to:", SAVE_PATH)


Saved compact H4 model to: C:\Users\admin\Desktop\Forex_algo\code\tft_h4-50.44.ckpt
